# Motor imagery decoding from EEG power and connectivity

PhysioNet EEGMMIDB participants 1–16 contribute 480 trials across runs 4 and 8. Three frequency bands and 21 sensorimotor electrodes yield 63 power and 630 correlation features. A four-fold outer split holds out entire people. Three inner participant folds tune logistic regularization. Each person receives equal weight in the reported metrics.

Published result: Mean participant AUROC was 0.632 for power alone, 0.520 for edges alone and 0.557 for fusion. The paired participant-bootstrap 95% interval for fusion minus power was −0.136 to −0.021. These sensor-level correlations come from EEG. The subset contains 16 participants, and the bootstrap conditions on the fitted folds.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'eeg-connectivity-decoding-real-data-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", 'numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2', 'edfio==0.4.18'], check=True)

### download.py

In [ ]:
(root / 'download.py').write_text('"""Cache the first 16 participants\' unilateral imagery recordings on the data drive."""\n\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nimport urllib.request\nimport time\n\nimport os\n\nDATA = Path(os.environ.get("EEG_DATA_DIR", Path(__file__).resolve().parent / "data"))\nBASE = "https://physionet.org/files/eegmmidb/1.0.0"\n\n\ndef fetch(key):\n    subject, run = key\n    tag = f"S{subject:03d}"\n    name = f"{tag}R{run:02d}.edf"\n    target = DATA / name\n    if target.exists():\n        return target\n    for attempt in range(3):\n        try:\n            with urllib.request.urlopen(f"{BASE}/{tag}/{name}", timeout=45) as response:\n                data = response.read()\n                assert len(data) == int(\n                    response.headers.get("Content-Length", len(data))\n                )\n                assert data.startswith(b"0       ") and len(data) > 1000000\n            temp = target.with_suffix(".part")\n            temp.write_bytes(data)\n            temp.replace(target)\n            return target\n        except OSError:\n            if attempt == 2:\n                raise\n            time.sleep(2)\n\n\nif __name__ == "__main__":\n    DATA.mkdir(parents=True, exist_ok=True)\n    with ThreadPoolExecutor(4) as pool:\n        for path in pool.map(fetch, [(i, r) for i in range(1, 17) for r in (4, 8)]):\n            print(path.name, flush=True)\n', encoding="utf8")

### train.py

In [ ]:
(root / 'train.py').write_text('"""Nested subject-held-out decoding from spectral power and EEG connectivity."""\n\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport edfio\nimport numpy as np\nfrom scipy.signal import butter, sosfiltfilt\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.model_selection import GroupKFold\nfrom sklearn.metrics import roc_auc_score, balanced_accuracy_score\nfrom download import DATA\n\nROOT = Path(__file__).resolve().parent\nCHANNELS = "fc5 fc3 fc1 fcz fc2 fc4 fc6 c5 c3 c1 cz c2 c4 c6 cp5 cp3 cp1 cpz cp2 cp4 cp6".split()\nBANDS = [(8, 13), (13, 20), (20, 30)]\n\n\ndef extract(path):\n    f = edfio.read_edf(path)\n    fs = f.signals[0].sampling_frequency\n    assert fs == 160 and all(s.sampling_frequency == fs for s in f.signals)\n    lookup = {s.label.strip(".").lower(): s.data for s in f.signals}\n    # Common average reference across all 64 electrodes.\n    signals = np.stack([lookup[k] for k in CHANNELS]) - np.stack(\n        list(lookup.values())\n    ).mean(0)\n    filters = [\n        sosfiltfilt(butter(4, band, btype="bandpass", fs=fs, output="sos"), signals)\n        for band in BANDS\n    ]\n    upper = np.triu_indices(len(CHANNELS), 1)\n    rows = []\n    for event in f.annotations:\n        if event.text not in ("T1", "T2"):\n            continue\n        assert event.duration >= 3.5\n        start = round((event.onset + 0.5) * fs)\n        power = []\n        edges = []\n        for band in filters:\n            epoch = band[:, start : start + int(3 * fs)]\n            assert epoch.shape[1] == 480\n            power.extend(np.log(np.maximum(epoch.var(1), 1e-12)))\n            edges.extend(np.arctanh(np.clip(np.corrcoef(epoch)[upper], -0.995, 0.995)))\n        rows.append((power, edges, int(event.text == "T2")))\n    return rows\n\n\ndef model(c):\n    return make_pipeline(\n        StandardScaler(), LogisticRegression(C=c, max_iter=800, class_weight="balanced")\n    )\n\n\ndef scores(y, p):\n    return {\n        "auroc": roc_auc_score(y, p),\n        "balanced_accuracy": balanced_accuracy_score(y, p >= 0.5),\n    }\n\n\ndef run(data):\n    power = []\n    edges = []\n    labels = []\n    subjects = []\n    files = {}\n    for subject in range(1, 17):\n        for recording in (4, 8):\n            path = data / f"S{subject:03d}R{recording:02d}.edf"\n            files[path.name] = hashlib.sha256(path.read_bytes()).hexdigest()\n            for p, e, y in extract(path):\n                power.append(p)\n                edges.append(e)\n                labels.append(y)\n                subjects.append(subject)\n    y, groups = np.array(labels), np.array(subjects)\n    methods = {\n        "power": np.array(power),\n        "edges": np.array(edges),\n        "fusion": np.c_[power, edges],\n    }\n    folds = []\n    individual = {}\n    metrics = {}\n    for name, x in methods.items():\n        preds = np.full(len(y), np.nan)\n        for fold, (train, test) in enumerate(GroupKFold(4).split(x, y, groups)):\n            assert not set(groups[train]) & set(groups[test])\n            inner = list(GroupKFold(3).split(x[train], y[train], groups[train]))\n            candidates = []\n            for c in (0.001, 0.01, 0.1, 1.0):\n                values = []\n                for a, b in inner:\n                    fit = model(c).fit(x[train[a]], y[train[a]])\n                    values.append(\n                        roc_auc_score(y[train[b]], fit.predict_proba(x[train[b]])[:, 1])\n                    )\n                candidates.append((np.mean(values), c))\n            _, c = max(candidates, key=lambda v: v[0])\n            fit = model(c).fit(x[train], y[train])\n            dest = ROOT / "checkpoints"\n            dest.mkdir(exist_ok=True)\n            scaler, lr = fit.steps[0][1], fit.steps[1][1]\n            np.savez_compressed(\n                dest / f"{name}-fold-{fold}.npz",\n                mean=scaler.mean_,\n                scale=scaler.scale_,\n                coef=lr.coef_,\n                intercept=lr.intercept_,\n                held_out=np.unique(groups[test]),\n            )\n            preds[test] = fit.predict_proba(x[test])[:, 1]\n            folds.append(\n                {\n                    "method": name,\n                    "fold": fold,\n                    "C": c,\n                    "held_out": np.unique(groups[test]).tolist(),\n                    **scores(y[test], preds[test]),\n                }\n            )\n        assert np.isfinite(preds).all()\n        np.savez_compressed(\n            ROOT / "checkpoints" / f"{name}-predictions.npz",\n            subject=groups,\n            y=y,\n            probability=preds,\n        )\n        individual[name] = {\n            int(g): scores(y[groups == g], preds[groups == g])\n            for g in np.unique(groups)\n        }\n        metrics[name] = {\n            k: np.mean([s[k] for s in individual[name].values()])\n            for k in ("auroc", "balanced_accuracy")\n        }\n    diff = np.array(\n        [\n            individual["fusion"][g]["auroc"] - individual["power"][g]["auroc"]\n            for g in individual["power"]\n        ]\n    )\n    boot = (\n        np.random.default_rng(17).choice(diff, (10000, len(diff)), replace=True).mean(1)\n    )\n    return {\n        "subjects": 16,\n        "epochs": len(y),\n        "right_fist_epochs": int(y.sum()),\n        "folds": folds,\n        "subject_metrics": individual,\n        "subject_macro": metrics,\n        "fusion_minus_power_auroc_subject_bootstrap_95": np.quantile(\n            boot, [0.025, 0.975]\n        ).tolist(),\n        "edf_sha256": files,\n    }\n\n\nif __name__ == "__main__":\n    cli = argparse.ArgumentParser()\n    cli.add_argument("--data", type=Path, default=DATA)\n    (ROOT / "results.json").write_text(json.dumps(run(cli.parse_args().data), indent=2))\n', encoding="utf8")

### Train and evaluate

In [ ]:
subprocess.run([sys.executable, 'download.py'], cwd=root, check=True)
subprocess.run([sys.executable, 'train.py'], cwd=root, check=True)

In [ ]:
import json
report = json.loads((root / "results.json").read_text())
print(json.dumps(report.get("means", report.get("subject_macro")), indent=2))